In [ ]:
# CELL 1: Imports e configuração

import logging
import pandas as pd

from functions.config import ProjectConfig
from functions.RefCif_Downloads import cif_download
from functions.Primary_Filter import primary_filter
from functions.Extraction_Phases import extrair_fracoes_fase
from functions.Workflow_GSAS2 import executar_refinamento_com_config
from functions.GSAS_Plots import plot_rietveld, plot_refinamento_com_referencias

# Configura logging para exibir mensagens de INFO e acima no notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

In [ ]:
# CELL 2: INPUTS e DIRETORIOS

# inputs
nome_do_projeto = "850_0.5H"
input_drx_raw = "inputs/HE850_0.5H.txt"
input_inst = "inputs/XRDynamic500_CuKa.instprm"

# Cria o diretório do projeto
cfg = ProjectConfig(nome_do_projeto)
cfg.create_dirs()
print(cfg.project_dir)
print(cfg.ref_dir)
print(cfg.results_dir)

In [ ]:
# CELL 3: Download referências

dict_refs = {
    # alpha-Fe2O3: Trigonal, R-3c. Referência clássica de alta cristalinidade.
    "Hematite": 2101168,

    # Fe3O4: Cúbico, Fd-3m. Estrutura de espinélio invertido.
    # Cuidado: Picos muito próximos aos da Maghemita.
    "Magnetite": 9005813,

    # gamma-Fe2O3: Cúbico, P4332. Fase metaestável.
    # Importante conferir se não há picos de superestrutura no seu DRX.
    "Maghemite": 9017489,

    # alpha-FeOOH: Ortorrômbico, Pnma. Comum se houver hidratação no resíduo.
    "Goethite": 7720752,

    # FeO: Cúbico, Fm-3m. FASE CRÍTICA para a redução a 850 °C.
    "Wüstite": 9002673,

    # Fe3C: Ortorrômbico, Pnma. Fase crítica para a redução a 850 °C.
    "Cementite": 9016584,

    # alpha-Fe: Cúbico, Im-3m. O produto final da redução (Ferro metálico).
    "Fe_metalico": 9008536,
}

for Nome, cod_ID in dict_refs.items():
    cif_download(Nome, cod_ID, str(cfg.ref_dir))

In [ ]:
# CELL 4: Filtragem primária

df, amostra_norm, theta = primary_filter(
    correlation="pearson",
    input_file=input_drx_raw,
    ref_dir=str(cfg.ref_dir),
)
print(df)

In [ ]:
# CELL 5: Plot

for i, row in df.iterrows():
    print(f"Referência: {row['nome']}, Score: {row['score']:.2%}")
    score = row['score']
    nome = row['nome']
    ref_int_norm = row['ref_int_norm']

    # Plot
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(theta, amostra_norm, label="Amostra (experimental)", linewidth=0.8)
    ax.plot(theta, ref_int_norm, label=f"Referência: {nome} (score={score:.2%})", linewidth=0.8, alpha=0.8)
    ax.set_xlabel("2θ (°)")
    ax.set_ylabel("Intensidade Normalizada")
    ax.set_title("XRD: Amostra vs Melhor Referência")
    ax.legend()
    plt.tight_layout()
    plt.show()   

    break

In [ ]:
cif_paths = [
    str(cfg.ref_dir / df['nome'][0]),
    #str(cfg.ref_dir / df['nome'][1]),
    #str(cfg.ref_dir / df['nome'][2]),
    #str(cfg.ref_dir / df['nome'][3]),
]
print(cif_paths)

In [ ]:
# PARAMETROS DE REFINAMEENTO

config_refinamento = {
    # --- Identificação e Caminhos ---
    "projeto": {
        "nome": cfg.project_name,
        "drx_path": input_drx_raw,
        "inst_path": input_inst,
        "cif_paths": cif_paths,
    },
    
    # --- Controle de Workflow (O que refinar?) ---
    "workflow": {
        "refinar_background": True,
        "refinar_escala": True,
        "refinar_deslocamento": True, # Sample Displacement
        "refinar_rede": True,         # Parâmetros de Célula (a, b, c)
        "refinar_perfil": True,       # Size e Mustrain
        "refinar_atomos": {
            "ativar": True,
            "etapas": ["U", "XU"]    # U = Térmico, XU = Posição + Térmico
        }
    },
    
    # --- Hiperparâmetros Técnicos ---
    "ajustes_tecnicos": {
        "max_cycles": 8,
        "size_model": "isotropic",
        "mustrain_model": "isotropic",
        "bkg_terms": 3,               # Número de termos de Chebyshev
    }
}


In [ ]:
# CELL 6: Refinamento Rietveld (GSAS-II)

resultado = executar_refinamento_com_config(config_refinamento)

In [ ]:
print(resultado["percentuais_fases"])
print(resultado["wR"])

In [ ]:
# CELL 7: Plot do Refinamento de Rietveld

plot_rietveld(resultado)

In [ ]:
# CELL 8: Plot do Refinamento com Referências

plot_refinamento_com_referencias(
    resultado["x"],
    resultado["yobs"],
    resultado["ycalc"],
    cif_paths,
    theta,
    nome_projeto=cfg.project_name
)

In [ ]:
# CELL 8: Extração automática dos percentuais de fase do .lst

lst_file_path = str(cfg.results_dir / f"{cfg.project_name}.lst")
print(f"Extraindo dados do arquivo: {lst_file_path}")

dados_extraidos = extrair_fracoes_fase(lst_file_path)

print("--- Resultados do Refinamento ---")
if isinstance(dados_extraidos, dict):
    for fase, fracoes in dados_extraidos.items():
        print(f"\nFase: {fase}")
        print(f"  Fração da Fase (Phase Fraction): {fracoes['Phase Fraction']}")

        weight_pct = fracoes['Weight Fraction'] * 100
        print(f"  Fração em Peso (Weight Fraction): {fracoes['Weight Fraction']} ({weight_pct:.2f}%)")
else:
    print(dados_extraidos)